### ROCm PyTorch on Docker Jupyter Container

In [4]:
import torch
print("PyTorch Version:", torch.__version__)
print("ROCm Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.9.1+rocm7.2.0.git7e1940d4
ROCm Available: True
GPU Name: AMD Radeon RX 9060 XT


### Test PyTorch on ROCm GPU with MNIST dataset

In [5]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if args.dry_run:
                break


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))


class Args(argparse.Namespace):
    batch_size = 64
    test_batch_size = 1000
    epochs = 1
    lr = 1.0
    gamma = 0.7
    no_accel = False
    dry_run = False
    seed = 1
    log_interval = 10
    save_model = True


def main():
    args=Args()
    use_accel = not args.no_accel and torch.accelerator.is_available()
    torch.manual_seed(args.seed)

    if use_accel:
        device = torch.accelerator.current_accelerator()
    else:
        device = torch.device("cpu")
    print(device)

    train_kwargs = {'batch_size': args.batch_size}
    test_kwargs = {'batch_size': args.test_batch_size}

    if use_accel:
        accel_kwargs = {'num_workers': 1,
                        'persistent_workers': True,
                       'pin_memory': True,
                       'shuffle': True}
        train_kwargs.update(accel_kwargs)
        test_kwargs.update(accel_kwargs)

    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
        ])
    
    dataset1 = datasets.MNIST('../data', train=True, download=True, transform=transform)
    dataset2 = datasets.MNIST('../data', train=False, transform=transform)
    train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
    test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

    model = Net().to(device)
    optimizer = optim.Adadelta(model.parameters(), lr=args.lr)

    scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)
    for epoch in range(1, args.epochs + 1):
        train(args, model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()

    if args.save_model:
        torch.save(model.state_dict(), "mnist_cnn.pt")

if __name__ == '__main__':
    main()

cuda
Train Epoch: 1 [0/60000 (0%)]	Loss: 2.295736
Train Epoch: 1 [640/60000 (1%)]	Loss: 1.118549
Train Epoch: 1 [1280/60000 (2%)]	Loss: 1.040174
Train Epoch: 1 [1920/60000 (3%)]	Loss: 0.824998
Train Epoch: 1 [2560/60000 (4%)]	Loss: 0.457279
Train Epoch: 1 [3200/60000 (5%)]	Loss: 0.426359
Train Epoch: 1 [3840/60000 (6%)]	Loss: 0.226194
Train Epoch: 1 [4480/60000 (7%)]	Loss: 0.719113
Train Epoch: 1 [5120/60000 (9%)]	Loss: 0.287263
Train Epoch: 1 [5760/60000 (10%)]	Loss: 0.125459
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.280583
Train Epoch: 1 [7040/60000 (12%)]	Loss: 0.175847
Train Epoch: 1 [7680/60000 (13%)]	Loss: 0.242896
Train Epoch: 1 [8320/60000 (14%)]	Loss: 0.125726
Train Epoch: 1 [8960/60000 (15%)]	Loss: 0.305983
Train Epoch: 1 [9600/60000 (16%)]	Loss: 0.255495
Train Epoch: 1 [10240/60000 (17%)]	Loss: 0.266914
Train Epoch: 1 [10880/60000 (18%)]	Loss: 0.200893
Train Epoch: 1 [11520/60000 (19%)]	Loss: 0.291768
Train Epoch: 1 [12160/60000 (20%)]	Loss: 0.084153
Train Epoch: 1 [12800/60